<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">

# Real-time credit scoring — the decision, not the score

*Draft notebook. The gateway ensembles two models and turns the PD into APPROVE / REFER / DECLINE with adverse-action reason codes. It runs two ways from the **`MODE`** parameter below: `local` scores the models in-process; `aws` invokes the deployed SageMaker endpoints. Same decision logic either way — the book's local↔AWS parity, in a notebook.*

<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">

<h2><i class="fas fa-cog" style="color:#18ab4b"></i>&nbsp; Parameters</h2>

Tagged `parameters` for papermill: run local, or flip to AWS with `papermill decision_service.ipynb out.ipynb -p MODE aws` (needs the endpoints up: `make -C aws push-models endpoints`).

In [1]:
# parameters
MODE = "local"  # "local": score in-process  |  "aws": invoke the SageMaker endpoints
REGION = "us-east-1"
ENDPOINTS = {"scorecard": "ch04-scorecard", "challenger": "ch04-challenger"}

<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">

<h2><i class="fas fa-random" style="color:#18ab4b"></i>&nbsp; One scoring call, two backends</h2>

`score(name, features)` is the only place the two modes differ: in `local` it calls the model's `probability_of_default` from the BYOC container source; in `aws` it calls `sagemaker-runtime`. Everything below is written against `score()` and does not care which.

In [2]:
import json

if MODE == "local":
    import importlib.util

    def _load(path, name):
        spec = importlib.util.spec_from_file_location(name, path)
        assert spec and spec.loader
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        return mod

    _models = {
        "scorecard": _load("../src/models/scorecard/model.py", "scorecard_model"),
        "challenger": _load("../src/models/challenger/model.py", "challenger_model"),
    }

    def score(name, features):
        """Score in-process, straight from the container's model.py."""
        return _models[name].probability_of_default(features)
else:
    import boto3

    _runtime = boto3.client("sagemaker-runtime", region_name=REGION)

    def score(name, features):
        """Score by invoking the model's SageMaker endpoint -- the gateway's path."""
        resp = _runtime.invoke_endpoint(
            EndpointName=ENDPOINTS[name],
            ContentType="application/json",
            Body=json.dumps(features),
        )
        return float(json.loads(resp["Body"].read())["pd"])


f"scoring backend: {MODE}"

'scoring backend: local'

<i aria-hidden="true" class="fas fa-clipboard-check" style="color:#18ab4b"></i> **Expected output:** a one-line confirmation string reporting which scoring backend is active:

```plain
'scoring backend: local'
```

<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">

<h2><i class="fas fa-code-branch" style="color:#18ab4b"></i>&nbsp; Ensemble — average the two probabilities</h2>

In [3]:
import pandas as pd


def ensemble(app):
    """Average the champion and challenger PDs -- the gateway's _ensemble()."""
    pds = {"scorecard": score("scorecard", app), "challenger": score("challenger", app)}
    return sum(pds.values()) / len(pds), pds


app = {
    "age": 40,
    "monthly_income": 6000,
    "requested_amount": 5000,
    "dti": 0.10,
    "utilization": 0.10,
}
pd_ensemble, model_pds = ensemble(app)
f"model PDs { ({k: round(v, 4) for k, v in model_pds.items()}) } -> ensemble {pd_ensemble:.4f}"

"model PDs {'scorecard': 0.0549, 'challenger': 0.0733} -> ensemble 0.0641"

<i aria-hidden="true" class="fas fa-clipboard-check" style="color:#18ab4b"></i> **Expected output:** the champion and challenger PDs for the sample application, followed by their averaged ensemble PD:

```plain
"model PDs {'scorecard': 0.0549, 'challenger': 0.0733} -> ensemble 0.0641"
```

<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">

<h2><i class="fas fa-balance-scale" style="color:#18ab4b"></i>&nbsp; The decision — hard rules, then the score cutoff</h2>

Hard policy rules first (each leaves an adverse-action reason code), then the ensemble PD against the approve/refer cutoffs. Mirrors `decide()` in `src/gateway/app.py`.

In [4]:
def decide(app):
    """Hard rules, then the ensemble cutoff -- the gateway's decision in one function."""
    reasons = []
    if not app.get("kyc_passed", True):
        reasons.append("KYC_FAILED")
    if app["age"] < 18:
        reasons.append("UNDER_MINIMUM_AGE")
    if app["dti"] > 0.50:
        reasons.append("DTI_TOO_HIGH")
    if app["requested_amount"] > 10 * app["monthly_income"]:
        reasons.append("AMOUNT_EXCEEDS_POLICY")

    pd_ensemble, model_pds = ensemble(app)
    if reasons:
        decision = "DECLINE"
    elif pd_ensemble < 0.10:
        decision = "APPROVE"
    elif pd_ensemble < 0.20:
        decision = "REFER"
        reasons.append("BORDERLINE_MANUAL_REVIEW")
    else:
        decision = "DECLINE"
        reasons.append("SCORE_BELOW_CUTOFF")
    return {
        "decision": decision,
        "pd": round(pd_ensemble, 4),
        "model_pds": {k: round(v, 4) for k, v in model_pds.items()},
        "score": round((1 - pd_ensemble) * 1000),
        "reasons": reasons,
    }


decide(app)

{'decision': 'APPROVE',
 'pd': 0.0641,
 'model_pds': {'scorecard': 0.0549, 'challenger': 0.0733},
 'score': 936,
 'reasons': []}

<i aria-hidden="true" class="fas fa-clipboard-check" style="color:#18ab4b"></i> **Expected output:** the full decision record for the sample application: an APPROVE with the ensemble PD, the per-model PDs, the 0-1000 score, and an empty reasons list:

```plain
{'decision': 'APPROVE',
 'pd': 0.0641,
 'model_pds': {'scorecard': 0.0549, 'challenger': 0.0733},
 'score': 936,
 'reasons': []}
```

<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">

<h2><i class="fas fa-clipboard-check" style="color:#18ab4b"></i>&nbsp; A batch of applications</h2>

In [5]:
applications = [
    {
        "id": "app-1042",
        "age": 40,
        "monthly_income": 6000,
        "requested_amount": 5000,
        "dti": 0.10,
        "utilization": 0.10,
    },
    {
        "id": "app-1043",
        "age": 34,
        "monthly_income": 4200,
        "requested_amount": 8000,
        "dti": 0.20,
        "utilization": 0.20,
    },
    {
        "id": "app-1044",
        "age": 34,
        "monthly_income": 4200,
        "requested_amount": 8000,
        "dti": 0.20,
        "utilization": 0.15,
        "kyc_passed": False,
    },
    {
        "id": "app-1099",
        "age": 30,
        "monthly_income": 4000,
        "requested_amount": 30000,
        "dti": 0.30,
        "utilization": 0.85,
    },
]
rows = []
for a in applications:
    d = decide({k: v for k, v in a.items() if k != "id"})
    rows.append(
        {"id": a["id"], **{k: d[k] for k in ("decision", "pd", "score", "reasons")}}
    )
pd.DataFrame(rows).set_index("id")

,decision,pd,score,reasons
id,,,,
app-1042,APPROVE,0.0641,936,[]
app-1043,REFER,0.1196,880,[BORDERLINE_MANUAL_REVIEW]
app-1044,DECLINE,0.1021,898,[KYC_FAILED]
app-1099,DECLINE,0.7463,254,[SCORE_BELOW_CUTOFF]


<i aria-hidden="true" class="fas fa-clipboard-check" style="color:#18ab4b"></i> **Expected output:** one row per application, each with its decision, PD, score, and any reason codes -- an approve, a borderline refer, a KYC decline, and a below-cutoff decline:

```plain
         decision      pd  score                     reasons
id                                                          
app-1042  APPROVE  0.0641    936                          []
app-1043    REFER  0.1196    880  [BORDERLINE_MANUAL_REVIEW]
app-1044  DECLINE  0.1021    898                [KYC_FAILED]
app-1099  DECLINE  0.7463    254        [SCORE_BELOW_CUTOFF]
```

---
*`MODE="aws"` runs this same notebook against the deployed endpoints (`make -C aws push-models endpoints` first); `make decide` calls the same decision through the gateway on Fargate/EKS. See `aws/`.*